# Credit Risk Scoring with Reject Inference

**Credit Risk · Data Cleaning · Logistic Regression · Reject Inference · Scorecard**

This project develops a credit-admission risk model for new card applicants.

The source data contains accepted, rejected and new applications. Only accepted customers have an observed default outcome.

## Key results

- Accepted applicants: **994**
- Rejected applicants: **291**
- New applicants: **34**
- Observed accepted default rate: **10.46%**
- Holdout ROC-AUC: **0.814**
- Holdout default recall: **0.885**
- Graded decision benchmark agreement: **85.7%**

The professional model deliberately excludes `Avgexp` and `Exp_Inc` because they are unavailable in a comparable way at application time for rejected and new applicants.

## 1. Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
)

## 2. Load the original workbook

The raw academic workbook is intentionally excluded from the public repository.  
To reproduce the analysis, place `DatosPractica_Scoring.xlsx` in `data/`.

In [2]:
data_path = Path("../data/DatosPractica_Scoring.xlsx")
df = pd.read_excel(data_path, sheet_name="datospracticas")
print("Shape:", df.shape)
df.head()

Shape: (1319, 14)


,ID,Cardhldr,default,Age,Income,Exp_Inc,Avgexp,Ownrent,Selfempl,Depndt,Inc_per,Cur_add,Major,Active
0,1,1.0,0.0,27.08333,2.400,0.016798,33.01333,0,0,0,2.400000,56,1,1
1,2,1.0,1.0,24.25000,3.500,0.069963,203.89170,0,0,0,3.500000,60,1,11
2,3,0.0,NaN,27.41667,1.600,0.000000,0.00000,1,0,1,0.800000,30,0,0
3,4,1.0,0.0,40.33333,3.067,0.159700,408.08250,0,0,2,1.022333,18,0,0
4,5,1.0,0.0,28.16667,3.350,0.071625,199.36920,1,0,0,3.350000,18,1,2


## 3. Separate applicant populations

In [3]:
accepted = df.loc[df["Cardhldr"] == 1].copy()
rejected = df.loc[df["Cardhldr"] == 0].copy()
new_applicants = df.loc[df["Cardhldr"].isna()].copy()

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
print("New applicants:", len(new_applicants))
print("Accepted defaults:", int(accepted["default"].sum()))
print("Accepted non-defaults:", int((accepted["default"] == 0).sum()))

Accepted: 994
Rejected: 291
New applicants: 34
Accepted defaults: 104
Accepted non-defaults: 890


## 4. Data-quality audit

Age contains several impossible or implausible values for a credit-card applicant.

The professional workflow treats ages `<18` or `>80` as invalid and lets the training pipeline impute them.

A more important issue appears in `Avgexp` and `Exp_Inc`: every rejected and every new applicant has a structural zero, while accepted customers generally have observed card expenditure. These variables are therefore excluded from the application-time model.

In [4]:
audit = pd.DataFrame({
    "population": ["accepted", "rejected", "new"],
    "rows": [len(accepted), len(rejected), len(new_applicants)],
    "age_below_18": [
        (accepted["Age"] < 18).sum(),
        (rejected["Age"] < 18).sum(),
        (new_applicants["Age"] < 18).sum(),
    ],
    "age_above_80": [
        (accepted["Age"] > 80).sum(),
        (rejected["Age"] > 80).sum(),
        (new_applicants["Age"] > 80).sum(),
    ],
    "avgexp_zero": [
        (accepted["Avgexp"] == 0).sum(),
        (rejected["Avgexp"] == 0).sum(),
        (new_applicants["Avgexp"] == 0).sum(),
    ],
    "exp_inc_zero": [
        (accepted["Exp_Inc"] == 0).sum(),
        (rejected["Exp_Inc"] == 0).sum(),
        (new_applicants["Exp_Inc"] == 0).sum(),
    ],
})
audit

,population,rows,age_below_18,age_above_80,avgexp_zero,exp_inc_zero
0,accepted,994,5,1,21,0
1,rejected,291,1,1,291,291
2,new,34,1,0,34,34


## 5. Application-time feature engineering

In [5]:
def build_application_matrix(data):
    out = pd.DataFrame(index=data.index)

    age = pd.to_numeric(data["Age"], errors="coerce")
    age = age.mask((age < 18) | (age > 80))

    income = pd.to_numeric(data["Income"], errors="coerce")
    inc_per = pd.to_numeric(data["Inc_per"], errors="coerce")
    cur_add = pd.to_numeric(data["Cur_add"], errors="coerce")
    active = pd.to_numeric(data["Active"], errors="coerce")

    out["Age"] = age
    out["log1p_Income"] = np.log1p(income.clip(lower=0))
    out["Ownrent"] = pd.to_numeric(data["Ownrent"], errors="coerce")
    out["Selfempl"] = pd.to_numeric(data["Selfempl"], errors="coerce")
    out["Depndt"] = pd.to_numeric(data["Depndt"], errors="coerce")
    out["log1p_Inc_per"] = np.log1p(inc_per.clip(lower=0))
    out["log1p_Cur_add"] = np.log1p(cur_add.clip(lower=0))
    out["Major"] = pd.to_numeric(data["Major"], errors="coerce")
    out["sqrt_Active"] = np.sqrt(active.clip(lower=0))

    return out

X = build_application_matrix(accepted)
y = accepted["default"].astype(int)
X.head()

,Age,log1p_Income,Ownrent,Selfempl,Depndt,log1p_Inc_per,log1p_Cur_add,Major,sqrt_Active
0,27.08333,1.223775,0,0,0,1.223775,4.043051,1,1.000000
1,24.25000,1.504077,0,0,0,1.504077,4.110874,1,3.316625
3,40.33333,1.402906,0,0,2,0.704252,2.944439,0,0.000000
4,28.16667,1.470176,1,0,0,1.470176,2.944439,1,1.414214
6,23.25000,1.056713,0,0,0,1.056713,2.564949,1,1.732051


## 6. Holdout design

Model selection is performed only on the development sample.  
The 25% holdout set is untouched until the model and threshold have been chosen.

In [6]:
idx_train, idx_test = train_test_split(
    np.arange(len(accepted)),
    test_size=0.25,
    random_state=42,
    stratify=y,
)

X_train = X.iloc[idx_train].to_numpy()
X_test = X.iloc[idx_test].to_numpy()
y_train = y.iloc[idx_train].to_numpy()
y_test = y.iloc[idx_test].to_numpy()

print("Development:", len(idx_train))
print("Holdout:", len(idx_test))

Development: 745
Holdout: 249


## 7. Regularization selection using training CV

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
C_grid = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]

rows = []

for C in C_grid:
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            C=C,
            random_state=42,
        )),
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
    )

    rows.append({
        "C": C,
        "cv_roc_auc_mean": scores.mean(),
        "cv_roc_auc_std": scores.std(),
    })

cv_results = pd.DataFrame(rows).sort_values("cv_roc_auc_mean", ascending=False)
cv_results

,C,cv_roc_auc_mean,cv_roc_auc_std
1,0.10,0.678851,0.076278
0,0.03,0.678314,0.077262
2,0.30,0.676999,0.075393
3,1.00,0.671548,0.075958
4,3.00,0.670713,0.076615
5,10.00,0.668988,0.076782


The selected regularization value is **C = 0.1**, based only on development cross-validation.

## 8. Development threshold from out-of-fold predictions

In [8]:
best_C = 0.1

base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        C=best_C,
        random_state=42,
    )),
])

oof_pd = cross_val_predict(
    base_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
)[:, 1]

fpr, tpr, thresholds = roc_curve(y_train, oof_pd)
threshold = thresholds[np.argmax(tpr - fpr)]

print("Development PD threshold:", round(float(threshold), 4))

Development PD threshold: 0.4692


## 9. Reject inference

A first model estimates risk for rejected applicants.  
Rejected cases are pseudo-labeled using the development threshold, then added to the development sample.

This is a **hard-cutoff reject inference** approach. It is transparent but assumption-dependent, so the notebook evaluates the final model only on accepted applicants with genuinely observed outcomes.

In [9]:
base_model.fit(X_train, y_train)

X_rejected = build_application_matrix(rejected).to_numpy()
rejected_pd = base_model.predict_proba(X_rejected)[:, 1]
rejected_pseudo = (rejected_pd >= threshold).astype(int)

print("Pseudo-default rejected:", int(rejected_pseudo.sum()))
print("Pseudo-non-default rejected:", int((rejected_pseudo == 0).sum()))

X_aug = np.vstack([X_train, X_rejected])
y_aug = np.concatenate([y_train, rejected_pseudo])

final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        C=best_C,
        random_state=42,
    )),
])

final_model.fit(X_aug, y_aug)

Pseudo-default rejected: 110
Pseudo-non-default rejected: 181


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

## 10. Holdout evaluation

In [10]:
holdout_pd = final_model.predict_proba(X_test)[:, 1]
holdout_pred = (holdout_pd >= threshold).astype(int)

metrics = {
    "ROC-AUC": roc_auc_score(y_test, holdout_pd),
    "Average Precision": average_precision_score(y_test, holdout_pd),
    "Balanced Accuracy": balanced_accuracy_score(y_test, holdout_pred),
    "Default Precision": precision_score(y_test, holdout_pred),
    "Default Recall": recall_score(y_test, holdout_pred),
    "Default F1": f1_score(y_test, holdout_pred),
}

pd.Series(metrics)

ROC-AUC              0.814074
Average Precision    0.318940
Balanced Accuracy    0.738272
Default Precision    0.201754
Default Recall       0.884615
Default F1           0.328571
dtype: float64

In [11]:
cm = confusion_matrix(y_test, holdout_pred)
cm

array([[132,  91],
       [  3,  23]])

The final model reaches **ROC-AUC 0.814** on the untouched holdout sample.

The default recall is **0.885**, which is important in credit risk because failing to identify a true default can be costly.

## 11. Score the 34 new applicants

In [12]:
X_new = build_application_matrix(new_applicants).to_numpy()
new_pd = final_model.predict_proba(X_new)[:, 1]

PDO = 40
BASE_SCORE = 600
BASE_ODDS = 50

factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)

def probability_to_score(probability):
    probability = np.clip(np.asarray(probability), 1e-6, 1 - 1e-6)
    odds = (1 - probability) / probability
    return offset + factor * np.log(odds)

scores = probability_to_score(new_pd)

applicant_scores = pd.DataFrame({
    "ID": new_applicants["ID"].astype(int).to_numpy(),
    "probability_of_default": new_pd,
    "credit_score": scores,
    "decision": np.where(new_pd < threshold, "APPROVE", "DECLINE"),
}).sort_values("ID")

applicant_scores

,ID,probability_of_default,credit_score,decision
0,1286,0.123609,487.277541,APPROVE
1,1287,0.842465,277.487807,DECLINE
2,1288,0.467668,381.719303,APPROVE
3,1289,0.088847,508.577337,APPROVE
4,1290,0.674697,332.147397,DECLINE
5,1291,0.539655,365.072868,DECLINE
6,1292,0.471666,380.793157,DECLINE
7,1293,0.641992,340.543196,DECLINE
8,1294,0.156408,471.495079,APPROVE
9,1295,0.898574,248.357213,DECLINE


## 12. Academic IV benchmark

The graded correction for the original Information Value question is preserved below as a benchmark.

It is intentionally separated from the professional application-time model because `Avgexp` and `Exp_Inc` are not available consistently for rejected or new applicants.

In [13]:
academic_iv = pd.DataFrame({
    "variable": [
        "Age", "Income", "Exp_Inc", "Avgexp", "Ownrent",
        "Selfempl", "Depndt", "Inc_per", "Cur_add", "Major", "Active",
    ],
    "graded_information_value": [
        0.224977, 0.222999, 0.187134, 0.271663, 0.011976,
        0.011895, 0.005121, 0.156240, 0.192857, 0.030557, 0.004177,
    ],
})
academic_iv

,variable,graded_information_value
0,Age,0.224977
1,Income,0.222999
2,Exp_Inc,0.187134
3,Avgexp,0.271663
4,Ownrent,0.011976
5,Selfempl,0.011895
6,Depndt,0.005121
7,Inc_per,0.156240
8,Cur_add,0.192857
9,Major,0.030557


## 13. Conclusions

### Main conclusions

- Credit applicants must be separated into accepted, rejected and new populations before modeling.
- The source data contains invalid age values that require cleaning.
- `Avgexp` and `Exp_Inc` behave as post-acceptance / structurally unavailable features and are excluded from the professional admission model.
- Model selection and threshold selection are performed without using the holdout set.
- Reject inference slightly changes the fitted risk model but cannot create true labels for rejected applicants; it remains an assumption-dependent technique.
- The final model produces a probability of default, score and approve/decline recommendation for all 34 new applicants.

### Limitations

- True outcomes for rejected applicants are not observed.
- The dataset is historical coursework data rather than a current production portfolio.
- Reject inference can reinforce historical approval bias.
- The threshold is an analytical threshold, not a regulatory or lender-approved credit policy.
- A production system would require fairness, stability, calibration, monitoring and governance controls.